In [2]:
!pip install pygame

In [1]:
import numpy
import pygame
import math
import time
from utils import *
from random import *
import sys


# The grass background auemntado 2.5 vezes mais
GRASS = scale_image(pygame.image.load("imgs/grass.jpg"), 2.5)
# The track image diminuida em 90%
TRACK = scale_image(pygame.image.load("imgs/track.png"), 0.9)

TRACK_MASK = scale_image(pygame.image.load("imgs/track-mask.png"), 0.9)

# The track border image for collision detection diminuída em 90%
TRACK_BORDER = scale_image(pygame.image.load("imgs/track-border.png"), 0.9)
TRACK_BORDER_MASK=pygame.mask.from_surface(TRACK_BORDER)

# The finish image intacta
FINISH = pygame.image.load("imgs/finish.png")
FINISH_MASK=pygame.mask.from_surface(FINISH)

# import cars images diminuídas em 55%
RED_CAR = scale_image(pygame.image.load("imgs/red-car.png"), 0.55)
GREEN_CAR = scale_image(pygame.image.load("imgs/green-car.png"), 0.55)
car_width,car_height=GREEN_CAR.get_size()
### half car-width and half car-height
HALF_WIDTH=car_width/2
HALF_HEIGHT=car_height/2
CAR_SIZE=HALF_WIDTH,HALF_HEIGHT

# Get the size of the track image
WIDTH, HEIGHT = TRACK.get_width(), TRACK.get_height()

# the window has the size of the track image
WIN = pygame.display.set_mode((WIDTH, HEIGHT))

# the name of the window
pygame.display.set_caption("Racing Game!")

pygame.font.init()
MAIN_FONT = pygame.font.SysFont("comicsans", 44)

FPS=60

RED = (255, 0, 0, 255)
WHITE = (255, 255, 255, 255)
YELLOW = (255, 255, 0, 255)

FINISH_POSITION=(130,250)
images=[(GRASS,(0,0)),(TRACK,(0,0)),(FINISH,FINISH_POSITION),(TRACK_BORDER,(0,0))]


def draw(win,images,player_car,computer_car,game_info):
    for img,pos in images:
        win.blit(img,pos)
    level_text=MAIN_FONT.render(f'Level {game_info.level}',1,(255,255,255))
    win.blit(level_text,(10,HEIGHT-level_text.get_height()-90))
    
    time_text=MAIN_FONT.render(f'Time {game_info.get_level_time()}',1,(255,255,255))
    win.blit(time_text,(10,HEIGHT-time_text.get_height()-50))
    
    velocity_text=MAIN_FONT.render(f'Vel {round(computer_car.vel,1)} px/s',1,(255,255,255))
    win.blit(velocity_text,(10,HEIGHT-velocity_text.get_height()-10))
    
    player_car.draw(win)
    computer_car.draw(win)
    pygame.display.update()
        
class AbstractCar:
    
    def __init__(self, max_vel, rotation_vel):
        self.img = self.IMG
        self.max_vel = max_vel
        self.vel = 0
        self.rotation_vel = rotation_vel
        self.angle = 0
        self.x,self.y=self.START_POS
        self.acceleration=1

    def rotate(self, left=False, right=False):
        if left and right:
            pass
        elif left:
            self.angle += self.rotation_vel + choice(range(-1,1))
        elif right:
            self.angle -= self.rotation_vel +  choice(range(-1,1))
        self.angle = int(self.angle) % 360
        
    def move_forward(self):
        self.vel = min(self.vel + self.acceleration, self.max_vel)
        self.move()
        
    def move_backwards(self):
        self.vel = max(self.vel - self.acceleration, -self.max_vel//2)
        self.move()
    
    
    # primeiro calculamos o  ângulo em radianos
    # Calculamos o deslocamento em x e y através da trignometra
    # actualizamos x e y mas subtraindo devido aos pontos cardeais do pygame e da corrida de carros
    def move(self):
        radians = math.radians(self.angle)
        vertical = math.cos(radians) * self.vel
        horizontal = math.sin(radians) * self.vel

        self.y -= vertical
        self.x -= horizontal
        
    
    def collide(self, mask, x=0, y=0):
        # 1. Roda a imagem atual com o ângulo atual do carro
        rotated_image = pygame.transform.rotate(self.img, self.angle)

        # 2. Mantém o centro da imagem rodada no mesmo sítio do centro da imagem original
        new_rect = rotated_image.get_rect(center=self.img.get_rect(topleft=(self.x, self.y)).center)

        # 3. Cria a máscara e calcula o offset com as novas coordenadas
        car_mask = pygame.mask.from_surface(rotated_image)
        offset = (int(new_rect.x - x), int(new_rect.y - y))

        poi = mask.overlap(car_mask, offset)
        return poi
        
    def reset(self):
        self.x,self.y=self.START_POS
        self.angle=0
        self.vel=0

    # x,y will be the center of the car
    #
    def draw(self, win):
        blit_rotate_center(win, self.img, (self.x, self.y), self.angle)
        #for b in self.bullets:
        #    b.draw(win)
        x, y = int(self.x + CAR_SIZE[0]), int(self.y + CAR_SIZE[1])
        dx, dy = CAR_SIZE[0], CAR_SIZE[1]
        alfa = (self.angle) * math.pi / 180
        mrot = numpy.array([[math.cos(alfa), -math.sin(alfa)], [math.sin(alfa), math.cos(alfa)]])
        pts = numpy.array([[-dx, -dy], [dx, -dy], [-dx, dy], [dx, dy]])
        npts = numpy.dot(pts, mrot)
        for i in npts:
            if 0<=i[0]+x<WIDTH and 0<=i[1]+y<HEIGHT: 
                pygame.draw.circle(win, TRACK_MASK.get_at((int(i[0] + x), int(i[1] + y))), \
                                                          (int(i[0] + x), int(i[1] + y)),2, 2)
        


class PlayerCar(AbstractCar):
    IMG = RED_CAR
    START_POS = (180, 200)
    
    def reduce_speed(self):
        self.vel = max(self.vel - self.acceleration / 2, 0)
        self.move()
        
    def bounce(self):
        self.vel=-self.vel
        self.move()
    
PATH=[(161, 148), (147, 82), (68, 93), (61, 174), (64, 251), (59, 464), (293, 711), (399, 712), (407, 537), (491, 474), (591, 529), (618, 707), (731, 709), (739, 383), (395, 334), (438, 250), (706, 251), (738, 96), (287, 93), (273, 395), (179, 399), (173, 258)]

        
class ComputerCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)
    
    def __init__(self, max_vel, rotation_vel, path=[]):
        super().__init__(max_vel, rotation_vel)
        self.path=path
        self.current_point=0
        self.vel=self.max_vel
             
    def calculate_angle(self):
        target_x, target_y = self.path[self.current_point]
        x_diff = target_x - self.x
        y_diff = target_y - self.y

        if y_diff == 0:
            desired_radian_angle = (1 if x_diff<0 else -1)*math.pi / 2
        else:
            desired_radian_angle = math.atan(x_diff / y_diff)

        if target_y > self.y:
            desired_radian_angle += math.pi

        difference_in_angle = self.angle - math.degrees(desired_radian_angle)
        if difference_in_angle >= 180:
            difference_in_angle -= 360
            
        elif difference_in_angle <= -180:
            difference_in_angle += 360

        if difference_in_angle > 0:
            self.angle -= min(self.rotation_vel, abs(difference_in_angle))
        else:
            self.angle += min(self.rotation_vel, abs(difference_in_angle))

    def update_path_point(self):
        target = self.path[self.current_point]
        rect = pygame.Rect(
            self.x, self.y, self.img.get_width(), self.img.get_height())
        if rect.collidepoint(*target):
            self.current_point += 1
            
    def move(self):
        if self.current_point >= len(self.path):
            return

        self.calculate_angle()
        self.update_path_point()
        super().move()
            
    def next_level(self,level):
        self.reset()
        self.vel=self.max_vel+(level-1)*0.02
        self.current_point=0
        
        
    def draw_points(self,win):
        for point in self.path:
            pygame.draw.circle(win,(255,0,0),point,5)
        
    def draw(self,win):
        super().draw(win)
        # self.draw_points(win)
        
        

def move_player(player_car):
    keys=pygame.key.get_pressed()
    moved=False
    if keys[pygame.K_a]:
        player_car.rotate(left=True)
    elif keys[pygame.K_d]:
        player_car.rotate(right=True)
    elif keys[pygame.K_w]:
        moved=True
        player_car.move_forward()
    elif keys[pygame.K_s]:
        moved=True
        player_car.move_backwards()
    if not moved:
        player_car.reduce_speed()


def handle_collision(player_car, computer_car,game_info):
    if player_car.collide(TRACK_BORDER_MASK) != None:
        player_car.bounce()

    computer_finish_poi_collide = computer_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if computer_finish_poi_collide != None:
        blit_text_center(WIN,MAIN_FONT,"YOU LOST!")
        pygame.display.update()
        pygame.time.wait(5000)
        game_info.reset()
        player_car.reset()
        computer_car.next_level(1)
        return True

    player_finish_poi_collide = player_car.collide(
        FINISH_MASK, *FINISH_POSITION)
    if player_finish_poi_collide != None:
        if player_finish_poi_collide[1] == 0:
            player_car.bounce()
        else:
            player_car.reset()
            game_info.next_level()
            computer_car.next_level(game_info.level)
            return True
    return False


class GameInfo:
    LEVELS = 10

    def __init__(self, level=1):
        self.level = level
        self.started = False
        self.level_start_time = 0

    def next_level(self):
        self.level += 1
        self.started = False

    def reset(self):
        self.level = 1
        self.started = False
        self.level_start_time = 0

    def game_finished(self):
        return self.level > self.LEVELS

    def start_level(self):
        self.started = True
        self.level_start_time = time.time()

    def get_level_time(self):
        if not self.started:
            return 0
        return round(time.time() - self.level_start_time)



pygame 2.6.1 (SDL 2.28.4, Python 3.12.7)
Hello from the pygame community. https://www.pygame.org/contribute.html


In [2]:
player_car=PlayerCar(4,4)
computer_car=ComputerCar(4,4,PATH)
game_info=GameInfo()
run=True
clock = pygame.time.Clock()

while run:
    clock.tick(FPS)
    draw(WIN,images,player_car,computer_car,game_info)
    
    while not game_info.started:
        blit_text_center(WIN, MAIN_FONT, f'Press any key to start level {game_info.level}!')
        pygame.display.update()
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                run=False
                pygame.quit()
                sys.exit()
            if event.type==pygame.KEYDOWN:
                game_info.start_level()
            
    
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run=False
            break
                            
    move_player(player_car)
    computer_car.move()
    
    if handle_collision(player_car, computer_car,game_info):
        draw(WIN,images,player_car,computer_car,game_info)
    
    if game_info.game_finished():
        blit_text_center(WIN,MAIN_FONT,"YOU WON!")
        pygame.time.wait(5000)
        game_info.reset()
        player_car.reset()  
        computer_car.next_level(1)
        
# pygame.quit()

SystemExit: 

d:\Anaconda\Lib\site-packages\IPython\core\interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Classe do Carro NEAT

Até temos duas classes de carros:

* Uma em que o carro é controlado pelo utilizador (com teclas).
* E outra em que o carro segue um percurso fixo.

Neste projeto temos como objetivo criar um carro que é controlado através de uma rede neuronal. Ou seja, vamos ter que criar uma nova classe `NeatCar` que aceite comandos numéricos contínuos da Rede Neuronal e precisamos de um ciclo de jogo (`game loop`) que treine dezenas de carros ao mesmo tempo.

In [3]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel):
        super().__init__(max_vel, rotation_vel)
        self.alive = True  # Para saber se o carro já bateu
        self.distance = 0  # Para a função de fitness
        self.current_waypoint = 0

    def apply_nn_actions(self, throttle, steering):
        """
        Recebe os outputs da rede neuronal.
        throttle: float entre -1 (travar) e 1 (acelerar no máximo)
        steering: float entre -1 (direita) e 1 (esquerda)
        """
        # --- REQUISITO 3.3: INJEÇÃO DE RUÍDO ---
        # Adicionamos uma pequena incerteza mecânica aos comandos da rede
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.1, 0.1)
        
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))

        # --- APLICAÇÃO DE FORÇAS ---
        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, -self.max_vel / 2) # Trava mais rápido do que acelera

        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360

        self.move()
        
        # Recompensa base: quanto mais rápido andar para a frente, melhor
        self.distance += self.vel 

    def check_collision(self):
        # Se bater na borda da pista, morre
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0
            
    def get_waypoint_data(self, waypoints):
        if len(waypoints) == 0:
            return [0, 0]
        
        target_x, target_y = waypoints[self.current_waypoint]
        
        # Distância euclidiana ao próximo waypoint (normalizada)
        dist = math.hypot(target_x - self.x, target_y - self.y)
        dist_normalizada = dist / math.hypot(WIDTH, HEIGHT)
        
        # Ângulo relativo entre a frente do carro e o waypoint
        # Mesma lógica do ComputerCar.calculate_angle()
        x_diff = target_x - self.x
        y_diff = target_y - self.y

        if y_diff == 0:
            desired_angle = (1 if x_diff < 0 else -1) * 90
        else:
            desired_angle = math.degrees(math.atan(x_diff / y_diff))

        if target_y > self.y:
            desired_angle += 180

        angle_diff = (self.angle - desired_angle) % 360
        if angle_diff > 180:
            angle_diff -= 360
        
        angle_normalizado = angle_diff / 180

        return [dist_normalizada, angle_normalizado]

    def update_waypoint(self, waypoints):
        target = waypoints[self.current_waypoint]
        rect = pygame.Rect(self.x, self.y, self.img.get_width(), self.img.get_height())
        if rect.collidepoint(*target):
            self.current_waypoint += 1
            if self.current_waypoint >= len(waypoints):
                self.current_waypoint = 0
    

Agora precisamos de um ciclo que nos permita treinar e valaivar várias instâncias do carro de uma só vez.

In [4]:
import neat
import os

In [107]:
def eval_genomes(genomes, config):
    nets = []
    cars = []
    ge = []

    # Inicializar os carros e as redes neuronais para esta geração
    for genome_id, genome in genomes:
        genome.fitness = 0  # Começam todos com 0 de fitness
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        cars.append(NeatCar(max_vel=4, rotation_vel=4))
        ge.append(genome)

    clock = pygame.time.Clock()
    run = True

    # O ciclo corre enquanto houver carros vivos na pista
    while run and len(cars) > 0:
        clock.tick(FPS)
        
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

        # Desenhar o fundo
        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))

        for i, car in enumerate(cars):
            # 1. OBTER INPUTS (Por agora são dummy/falsos. Mais tarde pomos radares ou waypoints)
            # Exemplo: dar a velocidade atual e o ângulo do carro
            inputs = (car.vel, car.angle) 
            
            # 2. PEDIR AÇÃO À REDE NEURONAL
            output = nets[i].activate(inputs)
            throttle = output[0]
            steering = output[1]

            # 3. MOVER O CARRO E VERIFICAR COLISÕES
            car.apply_nn_actions(throttle, steering)
            car.check_collision()

            # 4. ATUALIZAR FITNESS E REMOVER CARROS MORTOS
            if not car.alive:
                ge[i].fitness -= 10 # Penalização por bater
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
            else:
                ge[i].fitness += car.vel * 0.1 # Recompensa por andar depressa sem bater
                car.draw(WIN) # Só desenhamos os carros vivos

        pygame.display.update()
        
        # Condição de paragem de segurança (ex: fim de tempo limite por geração)
        # Podem implementar um contador de frames aqui para as gerações não ficarem infinitas

Agora só nos resta correr a função criada anteriormente. Para isso precisamos primeiro de criar um ficheiro de configuração, que se encontra na raiz do projeto.

In [34]:
def run_neat(config_path):
    pygame.init()
    
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game!")
    
    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)
    p = neat.Population(config)
    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)
    winner = p.run(eval_genomes, 50)
    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

run_neat('config-feedforward.txt')


 ****** Running generation 0 ****** 

Population's average fitness: -8.68753 stdev: 6.35459
Best fitness: 4.98162 - size: (2, 4) - species 1 - id 16
Average adjusted fitness: 0.561
Mean genetic distance 0.766, standard deviation 0.303
Population of 60 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0    60      4.982    0.561     0
Total extinctions: 0
Generation time: 1.975 sec

 ****** Running generation 1 ****** 

Population's average fitness: -1.56050 stdev: 6.54856
Best fitness: 4.65512 - size: (2, 4) - species 1 - id 36
Average adjusted fitness: 0.690
Mean genetic distance 0.812, standard deviation 0.311
Population of 60 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1    60      4.655    0.690     1
Total extinctions: 0
Generation time: 1.437 sec (1.706 average)

 ****** Running generation 2 **

SystemExit: 

Neste momento a rede neuronal está muito simples e "cega". Os pesos são atribuídos aleatoriamente, o que faz com que os carros tenham um comportamento estranho e apenas recebe 2 inputs: a velocidade e o ângulo.

### 3.2 Radares (sensores)

Para implementar estes radares vamos usar uma tecnologia chamada Raycasting. Esta tecnologia consiste em mandar raios, neste caso virtuais, a partir de vários ângulos e verificar se esses raios vão bater na borda da pista. No caso de baterem, medimos a distância percorrida e isso vai servir de input.
,
Para implementar isto, vamos atualizar a nossa classe `NeatCar` com novas funções (`check_radar,update_radars,get_data,draw_radars`)

In [10]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel, start_angle=0, track_reversed=False):
        super().__init__(max_vel, rotation_vel)
        self.angle = start_angle # Substitui o ângulo base (0) pelo novo
        self.track_reversed = track_reversed # Guarda a informação do sentido da pista
        if self.track_reversed:
            self.x, self.y = (175, 270) 
        else:
            self.x, self.y = (150, 200)
        self.alive = True
        self.distance = 0
        self.radars = [] # Lista para guardar os dados dos radares

        self.finished = False 
        self.finish_time = 0

    def check_radar(self, degree, track_border_mask):
        length = 0
        x = int(self.x + CAR_SIZE[0])
        y = int(self.y + CAR_SIZE[1])

        # Usar exatamente a mesma lógica do AbstractCar.move()
        # O carro original subtrai o sin() no X e o cos() no Y
        rad = math.radians(self.angle + degree)
        dx = -math.sin(rad)
        dy = -math.cos(rad)

        MAX_RADAR_LENGTH = 200 

        while length < MAX_RADAR_LENGTH:
            x = int(self.x + CAR_SIZE[0] + (dx * length))
            y = int(self.y + CAR_SIZE[1] + (dy * length))

            if x < 0 or x >= WIDTH or y < 0 or y >= HEIGHT:
                break
            
            # Se bater na parede, para o radar
            if track_border_mask.get_at((x, y)):
                break
                
            length += 1

        dist = int(math.sqrt(math.pow(x - (self.x + CAR_SIZE[0]), 2) + math.pow(y - (self.y + CAR_SIZE[1]), 2)))
        self.radars.append([(x, y), dist])

    def update_radars(self):
        """
        Limpa os radares antigos e lança 5 novos em ângulos diferentes.
        """
        self.radars.clear()
        # Lança 5 raios: -60º, -30º, 0º (frente), 30º, 60º
        for degree in [-60, -30, 0, 30, 60]:
            self.check_radar(degree, TRACK_BORDER_MASK)

    def get_data(self):
        """
        Retorna os inputs normalizados para a Rede Neuronal.
        """
        # Normalizamos a distância dividindo pelo comprimento máximo (200) 
        # para a rede receber valores entre 0.0 e 1.0 (muito mais fácil de treinar)
        return_values = [radar[1] / 200 for radar in self.radars]
        return return_values

    def draw_radars(self, win):
        """
        Desenha as linhas dos radares para poderes ver o carro a pensar.
        """
        for radar in self.radars:
            position = radar[0]
            pygame.draw.line(win, (0, 255, 0), (int(self.x + CAR_SIZE[0]), int(self.y + CAR_SIZE[1])), position, 1)
            pygame.draw.circle(win, (0, 255, 0), position, 3)

    def apply_nn_actions(self, throttle, steering):
        """
        Recebe os outputs da rede neuronal.
        throttle: float entre -1 (travar) e 1 (acelerar no máximo)
        steering: float entre -1 (direita) e 1 (esquerda)
        """
        # --- REQUISITO 3.3: INJEÇÃO DE RUÍDO ---
        # Adicionamos uma pequena incerteza mecânica aos comandos da rede
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.1, 0.1)
        
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))

        # --- APLICAÇÃO DE FORÇAS ---
        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            # TRAVA, MAS NÃO FAZ MARCHA-ATRÁS
            # Substituímos o limite negativo por 0
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, 0)

        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360

        self.move()
        
        # Recompensa base: quanto mais rápido andar para a frente, melhor
        self.distance += self.vel 

    def check_collision(self, frame_count):
        # 1. Bater na borda (morre)
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0

        # 2. Bater na meta (Vitória!)
        # NOVO: Só avaliamos a colisão com a meta após 3 segundos!
        # Impede que o cruzamento da meta no arranque (por causa da bounding box) mate o carro
        if frame_count > 180: 
            finish_poi_collide = self.collide(FINISH_MASK, *FINISH_POSITION)
            if finish_poi_collide != None:
                # Lógica original: bater de frente dá y > 0. Bater por trás dá y == 0.
                contra_mao = (finish_poi_collide[1] == 0)
                
                # Se a pista estiver invertida, a regra do contra-mão inverte-se!
                if self.track_reversed:
                    contra_mao = not contra_mao

                if contra_mao:
                    self.alive = False
                    self.vel = 0
                else:
                    self.finished = True
                    self.vel = 0
                    self.finish_time = frame_count / FPS
    
    # IMPORTANTE: Garante que o teu draw original também chama os radares
    def draw(self, win):
        super().draw(win)
        self.draw_radars(win)

Com esta nova classe criada, agora precisamos de atualizar a forma como os inputs são passados à rede neuronal, pois agora precisamos de passar os 5 radares. Para isso atualizamos a função `eval_genomes`. 

**Nota**: Também tem que se mudar no ficheiro de configuração o número de inputs passados à rede neuronal.

In [11]:
def eval_genomes(genomes, config):
    # --- INTERRUPTOR DE PISTA ---
    # Muda para False para correr normal, ou True para correr ao contrário
    INVERTER_PISTA = False 
    ANGULO_INICIAL = 180 if INVERTER_PISTA else 0
    
    nets = []
    cars = []
    ge = []

    for genome_id, genome in genomes:
        genome.fitness = 0 
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        # Passamos os novos parâmetros na criação do carro
        cars.append(NeatCar(max_vel=4, rotation_vel=4, start_angle=ANGULO_INICIAL, track_reversed=INVERTER_PISTA))
        ge.append(genome)

    clock = pygame.time.Clock()
    run = True
    frame_count = 0

    # O ciclo corre enquanto houver carros vivos na pista
    while run and len(cars) > 0:
        clock.tick(FPS)
        frame_count += 1 # Aumenta a cada frame

        # Se a geração demorar demasiado tempo, matamos todos os carros que sobram
        if frame_count > 20000:
            break
            
        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()

        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))
        WIN.blit(FINISH, FINISH_POSITION)

        for i in reversed(range(len(cars))):
            car = cars[i]
            
            car.update_radars()
            inputs = car.get_data() 
            output = nets[i].activate(inputs)
            
            throttle = output[0]
            steering = output[1]

            # --- NOVO: ZONA MORTA (ESTABILIZADOR DE DIREÇÃO) ---
            # Se a IA pedir para virar menos de 20%, ignoramos e forçamos o volante a direito (0).
            # Isto permite que os carros consigam fazer a reta perfeitamente a direito sem drift!
            if abs(steering) < 0.2:
                steering = 0
            
            car.apply_nn_actions(throttle, steering)
            car.check_collision(frame_count)

            if INVERTER_PISTA:
                # O carro só morre se tentar subir (y < 220) E ainda estiver na zona da reta da meta (x < 300)
                if car.y < 220 and car.x < 200 and car.x > 150:
                    car.alive = False

            # --- NOVO: CHECKPOINT ANTI-PIÕES ---
            # Aos 3 segundos exatos (180 frames), obriga o carro a ter saído do sítio!
            if frame_count == 180:
                # Calcula a distância em linha reta desde a NOVA posição de spawn (270 em vez de 230)
                dist_start = math.hypot(car.x - 175, car.y - 270) if INVERTER_PISTA else math.hypot(car.x - 150, car.y - 200)
                if dist_start < 100: # Se andou menos de 100 pixeis em 3 segundos...
                    car.alive = False

            if car.finished:
                # 1. Dá um fitness absurdamente alto para ultrapassar o "fitness_threshold" do config
                ge[i].fitness += 10000 
                
                # 2. Desenha a mensagem de vitória e o tempo no ecrã do Pygame
                mensagem = MAIN_FONT.render("O AI VENCEU!", 1, (255, 255, 255))
                tempo_txt = MAIN_FONT.render(f"Tempo: {car.finish_time:.2f} s", 1, (255, 255, 0)) # Amarelo
                
                WIN.blit(mensagem, (WIDTH//2 - mensagem.get_width()//2, HEIGHT//2 - 50))
                WIN.blit(tempo_txt, (WIDTH//2 - tempo_txt.get_width()//2, HEIGHT//2 + 10))
                pygame.display.update()
                
                print(f"\n[!] SUCESSO! O carro da geração encontrou a meta em {car.finish_time:.2f} segundos!")
                
                # 3. Congela o ecrã durante 4 segundos para vocês poderem festejar e ler o tempo
                pygame.time.wait(4000) 
                
                # 4. Esvazia a lista de carros para forçar o fim imediato desta geração
                cars.clear()
                break # Sai do ciclo 'for'
                
           # MATA se bater ou ficar parado, MAS SEM TIRAR PONTOS
            elif not car.alive or (car.vel <= 0 and frame_count > 60):
                # APAGÁMOS O ge[i].fitness -= 10
                # O carro simplesmente morre e não ganha mais pontos
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
                
            else:
                # Se for a direito (volante 0), ganha 100% dos pontos. 
                # Se trancar o volante (1 ou -1), o abs(steering) fica 1, logo 1.0 - 1.0 = 0. Não ganha pontos!
                ge[i].fitness += car.vel
                car.draw(WIN)

        pygame.display.update()
        
        # Condição de paragem de segurança (ex: fim de tempo limite por geração)
        # Podem implementar um contador de frames aqui para as gerações não ficarem infinitas

E corremos agora a função nova...

In [12]:
def run_neat(config_path):
    pygame.init()
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game!")
    
    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)

    p = neat.Population(config)

    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    winner = p.run(eval_genomes, 100)

    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

run_neat('config-feedforward.txt')


 ****** Running generation 0 ****** 

Population's average fitness: 40.58341 stdev: 134.58327
Best fitness: 1417.23734 - size: (2, 10) - species 1 - id 115
Average adjusted fitness: 0.029
Mean genetic distance 1.405, standard deviation 0.345
Population of 120 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0   120   1417.237    0.029     0
Total extinctions: 0
Generation time: 6.078 sec

 ****** Running generation 1 ****** 

Population's average fitness: 85.90638 stdev: 186.83090
Best fitness: 1401.75554 - size: (2, 10) - species 1 - id 162
Average adjusted fitness: 0.061
Mean genetic distance 1.339, standard deviation 0.313
Population of 120 members in 1 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1   120   1401.756    0.061     1
Total extinctions: 0
Generation time: 5.993 sec (6.035 average)

 ****** Running

: 

### 3.3 Waypoints

Para implementar a abordagem de waypoints, o utilizador vai marcar pontos de referência ao longo da pista com o rato. Esses pontos definem o percurso ideal que o carro deve seguir.

A rede neuronal não segue os waypoints diretamente. Em vez disso, recebe como inputs a distância e o ângulo até ao próximo waypoint e aprende ela própria a conduzir em direção a esse ponto.

Para implementar isto, vamos criar uma função que captura os cliques do rato e atualizar a classe `NeatCar` com novos métodos para calcular os inputs relativos ao waypoint atual.

In [5]:
pygame.init()
global WIN
WIN = pygame.display.set_mode((WIDTH, HEIGHT))
pygame.display.set_caption("Marca os Waypoints!")

waypoints = []
run = True
clock = pygame.time.Clock()

print("Clica na pista para adicionar waypoints. Prime ENTER para terminar.")

while run:
    clock.tick(FPS)
    
    WIN.blit(GRASS, (0, 0))
    WIN.blit(TRACK, (0, 0))

    for i, point in enumerate(waypoints):
        pygame.draw.circle(WIN, (255, 0, 0), point, 6)
        if i > 0:
            pygame.draw.line(WIN, (255, 0, 0), waypoints[i-1], point, 2)

    pygame.display.update()

    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            run = False
            pygame.quit()
            sys.exit()

        if event.type == pygame.MOUSEBUTTONDOWN:
            pos = pygame.mouse.get_pos()
            waypoints.append(pos)
            print(f"Waypoint {len(waypoints)}: {pos}")

        if event.type == pygame.KEYDOWN:
            if event.key == pygame.K_RETURN:
                run = False

print(f"\nWaypoints guardados: {waypoints}")

Clica na pista para adicionar waypoints. Prime ENTER para terminar.
Waypoint 1: (169, 165)
Waypoint 2: (162, 105)
Waypoint 3: (124, 81)
Waypoint 4: (72, 93)
Waypoint 5: (58, 132)
Waypoint 6: (58, 199)
Waypoint 7: (58, 305)
Waypoint 8: (63, 422)
Waypoint 9: (86, 493)
Waypoint 10: (132, 555)
Waypoint 11: (186, 615)
Waypoint 12: (245, 669)
Waypoint 13: (293, 705)
Waypoint 14: (334, 723)
Waypoint 15: (370, 721)
Waypoint 16: (393, 703)
Waypoint 17: (408, 665)
Waypoint 18: (399, 625)
Waypoint 19: (397, 584)
Waypoint 20: (393, 552)
Waypoint 21: (412, 516)
Waypoint 22: (483, 475)
Waypoint 23: (524, 470)
Waypoint 24: (570, 487)
Waypoint 25: (598, 516)
Waypoint 26: (607, 550)
Waypoint 27: (604, 598)
Waypoint 28: (601, 633)
Waypoint 29: (602, 676)
Waypoint 30: (617, 715)
Waypoint 31: (641, 733)
Waypoint 32: (700, 728)
Waypoint 33: (732, 691)
Waypoint 34: (739, 645)
Waypoint 35: (739, 592)
Waypoint 36: (743, 536)
Waypoint 37: (741, 487)
Waypoint 38: (740, 442)
Waypoint 39: (725, 398)
Waypoint 40: 

In [6]:
class NeatCar(AbstractCar):
    IMG = GREEN_CAR
    START_POS = (150, 200)

    def __init__(self, max_vel, rotation_vel, start_angle=0, track_reversed=False):
        super().__init__(max_vel, rotation_vel)
        self.angle = start_angle
        self.x, self.y = (150, 200)
        self.alive = True
        self.distance = 0
        self.current_waypoint = 0
        self.finished = False
        self.finish_time = 0

    def get_waypoint_data(self, waypoints):
        if len(waypoints) == 0:
            return [0, 0]
        
        target_x, target_y = waypoints[self.current_waypoint]
        
        # Input 1: Distância normalizada
        dist = math.hypot(target_x - self.x, target_y - self.y)
        dist_normalizada = dist / math.hypot(WIDTH, HEIGHT)
        
        # Input 2: Ângulo relativo ao próximo waypoint
        x_diff = target_x - self.x
        y_diff = target_y - self.y
        if y_diff == 0:
            desired_angle = (1 if x_diff < 0 else -1) * 90
        else:
            desired_angle = math.degrees(math.atan(x_diff / y_diff))
        if target_y > self.y:
            desired_angle += 180
        angle_diff = (self.angle - desired_angle) % 360
        if angle_diff > 180:
            angle_diff -= 360
        angle_normalizado = -angle_diff / 180

        return [dist_normalizada, angle_normalizado]

    def update_waypoint(self, waypoints):
        target_x, target_y = waypoints[self.current_waypoint]
        dist = math.hypot(target_x - self.x, target_y - self.y)
        if dist < 25:
            self.current_waypoint += 1
            if self.current_waypoint >= len(waypoints):
                self.current_waypoint = 0

    def apply_nn_actions(self, throttle, steering):
        noise_throttle = uniform(-0.05, 0.05)
        noise_steering = uniform(-0.05, 0.05)
        throttle = max(-1.0, min(1.0, throttle + noise_throttle))
        steering = max(-1.0, min(1.0, steering + noise_steering))
        if throttle > 0:
            self.vel = min(self.vel + self.acceleration * throttle, self.max_vel)
        else:
            self.vel = max(self.vel + (self.acceleration * 2) * throttle, 0)
        self.angle += self.rotation_vel * steering
        self.angle = int(self.angle) % 360
        self.move()
        self.distance += self.vel

    def check_collision(self, frame_count):
        if self.collide(TRACK_BORDER_MASK) != None:
            self.alive = False
            self.vel = 0
        if frame_count > 180:
            finish_poi_collide = self.collide(FINISH_MASK, *FINISH_POSITION)
            if finish_poi_collide != None:
                if finish_poi_collide[1] == 0:
                    self.alive = False
                    self.vel = 0
                else:
                    self.finished = True
                    self.vel = 0
                    self.finish_time = frame_count / FPS

    def draw(self, win):
        super().draw(win)

In [7]:
def eval_genomes_waypoints(genomes, config):
    INVERTER_PISTA = False
    ANGULO_INICIAL = 180 if INVERTER_PISTA else 0

    nets = []
    cars = []
    ge = []

    for genome_id, genome in genomes:
        genome.fitness = 0
        net = neat.nn.FeedForwardNetwork.create(genome, config)
        nets.append(net)
        cars.append(NeatCar(max_vel=4, rotation_vel=5.5, start_angle=ANGULO_INICIAL, track_reversed=INVERTER_PISTA))
        ge.append(genome)

    clock = pygame.time.Clock()
    frame_count = 0

    selected_waypoint = None
    while len(cars) > 0:
        clock.tick(FPS)
        frame_count += 1

        if frame_count > 20000:
            break

        for event in pygame.event.get():
            if event.type == pygame.QUIT:
                pygame.quit()
                sys.exit()
            
            if event.type == pygame.MOUSEBUTTONDOWN:
                mouse_x, mouse_y = pygame.mouse.get_pos()
                for j, point in enumerate(waypoints):
                    if math.hypot(mouse_x - point[0], mouse_y - point[1]) < 15:
                        selected_waypoint = j
                        break
            
            if event.type == pygame.MOUSEBUTTONUP:
                selected_waypoint = None
            
            if event.type == pygame.MOUSEMOTION and selected_waypoint is not None:
                waypoints[selected_waypoint] = pygame.mouse.get_pos()

        WIN.blit(GRASS, (0, 0))
        WIN.blit(TRACK, (0, 0))
        WIN.blit(FINISH, FINISH_POSITION)

        for point in waypoints:
            pygame.draw.circle(WIN, (255, 0, 0), point, 5)

        for i in reversed(range(len(cars))):
            car = cars[i]

            old_waypoint = car.current_waypoint
            car.update_waypoint(waypoints)
            inputs = car.get_waypoint_data(waypoints)
            output = nets[i].activate(inputs)

            throttle = output[0]
            steering = output[1]

            if abs(steering) < 0.2:
                steering = 0

            car.apply_nn_actions(throttle, steering)
            car.check_collision(frame_count)

            if frame_count == 180:
                dist_start = math.hypot(car.x - 150, car.y - 200)
                if dist_start < 100:
                    car.alive = False

            if car.finished:
                ge[i].fitness += 10000
                print(f"\n[!] SUCESSO! O carro encontrou a meta em {car.finish_time:.2f} segundos!")
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)
                break

            elif not car.alive or (car.vel <= 0 and frame_count > 60):
                cars.pop(i)
                nets.pop(i)
                ge.pop(i)

            else:
                if car.current_waypoint != old_waypoint:
                    ge[i].fitness += 200

                car.draw(WIN)
                wp_font = pygame.font.SysFont("comicsans", 20)
                wp_txt = wp_font.render(str(car.current_waypoint), 1, (255, 255, 0))
                WIN.blit(wp_txt, (int(car.x), int(car.y)))

        pygame.display.update()

In [ ]:
def run_neat_waypoints(config_path):
    pygame.init()
    global WIN
    WIN = pygame.display.set_mode((WIDTH, HEIGHT))
    pygame.display.set_caption("Racing Game - Waypoints!")

    config = neat.config.Config(neat.DefaultGenome, neat.DefaultReproduction,
                                neat.DefaultSpeciesSet, neat.DefaultStagnation,
                                config_path)

    p = neat.Population(config)

    p.add_reporter(neat.StdOutReporter(True))
    stats = neat.StatisticsReporter()
    p.add_reporter(stats)

    import time
    start_time = time.time()
    winner = p.run(eval_genomes_waypoints, 25)
    elapsed_time = time.time() - start_time

    pygame.quit()

    try:
        import pickle
        with open('winner_waypoints.pkl', 'wb') as f:
            pickle.dump(winner, f)

        info = {
            'melhor_fitness': winner.fitness,
            'melhor_geracao': stats.best_genome_generation(),
            'tempo_total': elapsed_time,
            'num_waypoints': len(waypoints),
            'config': config_path
        }
        with open('stats_waypoints.pkl', 'wb') as f:
            pickle.dump(info, f)

        print(f"\nMelhor fitness: {winner.fitness}")
        print(f"Melhor geração: {stats.best_genome_generation()}")
        print(f"Tempo total: {elapsed_time:.1f} segundos")
        print(f"Número de waypoints: {len(waypoints)}")
    except Exception as e:
        print(f"Erro ao guardar: {e}")

    print('\nMelhor genoma encontrado:\n{!s}'.format(winner))

run_neat_waypoints('config-feedforward.txt')


 ****** Running generation 0 ****** 

Population's average fitness: 75.00000 stdev: 177.12990
Best fitness: 1200.00000 - size: (2, 4) - species 2 - id 76
Average adjusted fitness: 0.070
Mean genetic distance 2.023, standard deviation 0.561
Population of 120 members in 2 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    0    69    400.000    0.056     0
     2    0    51   1200.000    0.083     0
Total extinctions: 0
Generation time: 1.708 sec

 ****** Running generation 1 ****** 

Population's average fitness: 235.00000 stdev: 505.74203
Best fitness: 4200.00000 - size: (2, 4) - species 2 - id 189
Average adjusted fitness: 0.059
Mean genetic distance 1.988, standard deviation 0.513
Population of 120 members in 2 species (after reproduction):
   ID   age  size   fitness   adj fit  stag
  ====  ===  ====  =========  =======  ====
     1    1    27   2000.000    0.039     0
     2    1    93   4200.000    0.079 

In [13]:
import pickle

# Ver o winner
with open('winner_waypoints.pkl', 'rb') as f:
    winner = pickle.load(f)
print(winner)

# Ver as stats
with open('stats_waypoints.pkl', 'rb') as f:
    info = pickle.load(f)
print(info)

Key: 1675
Fitness: 24200
Nodes:
	0 DefaultNodeGene(key=0, bias=0.35955162258350337, response=1.0, activation=tanh, aggregation=sum, time_constant=1.0)
	1 DefaultNodeGene(key=1, bias=0.07854838254358809, response=1.0, activation=tanh, aggregation=sum, time_constant=1.0)
Connections:
	DefaultConnectionGene(key=(-2, 1), innovation=4, weight=3.2941981983220736, enabled=True)
	DefaultConnectionGene(key=(-1, 0), innovation=1, weight=-0.9246396051002106, enabled=True)
	DefaultConnectionGene(key=(-1, 1), innovation=2, weight=-2.0565003614585233, enabled=False)
{'melhor_fitness': 24200, 'num_waypoints': 73, 'config': 'config-feedforward.txt'}
